# Lab 03 · Pandas for Data Manipulation and Cleaning
**CS-3105 Machine Learning Lab** · Lab Instructor: Engr. Muhammad Awais  
**CLO 6:** Analyze real-world datasets by applying data manipulation and visualization techniques.  

**Student:** Muhammad Hasham Munir Qazi  
**Roll No:** 2024-SE-32

**Dataset:** Pakistan used cars. Keep `pakistan_used_cars.csv` (the original instructor file) or `pakistan_used_cars_clean.csv` (the cleaned file supplied with this notebook) in the same folder. The notebook automatically uses the original file when it is available; otherwise, it uses the cleaned file. Run the notebook from top to bottom.

## Task 1 · Load and first look
- (a) Read the dataset and print its shape.
- (b) Show the first 5 rows.
- (c) Print the column names.
- (d) Show the data type of every column.
- (e) Identify the price column's data type and show example values.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", None)

raw_path = Path("pakistan_used_cars.csv")
clean_path = Path("pakistan_used_cars_clean.csv")

if raw_path.exists():
    df = pd.read_csv(raw_path)
    source_type = "original"
elif clean_path.exists():
    df = pd.read_csv(clean_path)
    source_type = "already-cleaned"
else:
    raise FileNotFoundError(
        "Dataset not found. Put pakistan_used_cars.csv or "
        "pakistan_used_cars_clean.csv in the same folder as this notebook."
    )

required_columns = ["Brand", "Model", "Price", "Registered City", "Fuel",
                    "Transaction Type", "Year", "KMs Driven"]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"The dataset is missing required columns: {missing_columns}")

print("Dataset source:", source_type)
print("T1a - Dataset shape:", df.shape)
print("T1b - First five rows:")
display(df.head())
print("T1c - Column names:", df.columns.tolist())
print("T1d - Column data types:")
print(df.dtypes)
print("T1e - Price examples:")
print(df["Price"].head(2))
if pd.api.types.is_numeric_dtype(df["Price"]):
    print("Price is already numeric in the supplied cleaned dataset.")
else:
    text_prices = df.loc[df["Price"].astype("string").str.contains(r"Rs|,", na=False), "Price"].head(2)
    print("Price is stored as text. Examples containing Rs or commas:")
    display(text_prices)

## Task 2 · Retrieve data
- (a) First 5 values of `Brand`.
- (b) First 5 rows of `Brand`, `Model`, and `Price`.
- (c) First row using `iloc`.
- (d) `Model` at index 100 using `loc` (when that index exists).
- (e) Number of cars registered in Islamabad.
- (f) Number of Honda cars using Hybrid fuel.
- (g) Cars from 2018 or newer, newest first (first 10).
- (h) Explain the comparison issue when `Price` is text.

In [ ]:
print("T2a - First five brands:")
display(df["Brand"].head())

print("T2b - Brand, Model and Price:")
display(df[["Brand", "Model", "Price"]].head())

print("T2c - First row:")
if not df.empty:
    display(df.iloc[0])
else:
    print("The dataset has no rows.")

print("T2d - Model at index 100:")
if 100 in df.index:
    print(df.loc[100, "Model"])
else:
    print("Index 100 is not present in this dataset.")

print("T2e - Islamabad advertisements:",
      df["Registered City"].astype("string").str.strip().str.casefold().eq("islamabad").sum())
print("T2f - Honda Hybrid advertisements:",
      (df["Brand"].astype("string").str.casefold().eq("honda") &
       df["Fuel"].astype("string").str.casefold().eq("hybrid")).sum())

print("T2g - Cars from 2018 onward:")
year_values = pd.to_numeric(df["Year"], errors="coerce")
newer_cars = df.loc[year_values.ge(2018), ["Brand", "Model", "Year", "Price"]].copy()
newer_cars["Year"] = pd.to_numeric(newer_cars["Year"], errors="coerce")
display(newer_cars.sort_values("Year", ascending=False).head(10))

print("T2h - Price comparison:")
if pd.api.types.is_numeric_dtype(df["Price"]):
    print("Price is numeric in this input, so comparison with 1,000,000 works:")
    display(df.loc[pd.to_numeric(df["Price"], errors="coerce").gt(1_000_000)].head())
else:
    print("A text Price column cannot be compared directly with an integer. "
          "Remove Rs and commas, then convert Price to a numeric type before comparison.")

## Task 3 · Clean the data
- (a) Count missing values. Drop rows without `Brand` or `Model`; fill missing city, fuel, and transaction type with `Unknown`.
- (b) Standardize city formatting and remove `Rs` and commas from `Price`.
- (c) Convert `Price` to integer and `Year` to nullable integer.
- (d) Correct `Sukkar` and `Corrolla Altis` spelling.
- (e) Count and remove exact duplicate rows.
- (f) Mark invalid mileage values as missing and remove the specified fake prices.
- (g) Display the final cleaned shape.

When the cleaned CSV is used as input, this section safely repeats the cleaning steps without requiring the unavailable original raw file.

In [ ]:
print("T3a - Missing values by column:")
print(df.isna().sum())

rows_before_missing = len(df)
df = df.dropna(subset=["Brand", "Model"]).copy()
for column in ["Registered City", "Fuel", "Transaction Type"]:
    df[column] = df[column].fillna("Unknown")
print("T3a - Rows removed because Brand or Model was missing:", rows_before_missing - len(df))

# Standardize text fields and price formatting.
df["Registered City"] = df["Registered City"].astype("string").str.strip().str.title()
city_categories_before = df["Registered City"].nunique(dropna=True)
price_text = (df["Price"].astype("string")
              .str.replace("Rs", "", regex=False)
              .str.replace(",", "", regex=False)
              .str.strip())
price_numeric = pd.to_numeric(price_text, errors="coerce")
invalid_price_rows = int(price_numeric.isna().sum())
if invalid_price_rows:
    print("T3b - Rows removed because Price was missing or not numeric:", invalid_price_rows)
    df = df.loc[price_numeric.notna()].copy()
    price_numeric = price_numeric.loc[price_numeric.notna()]
df["Price"] = price_numeric.astype("int64")

# Preserve missing Year values using pandas nullable integer dtype.
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
print("T3c - Price dtype:", df["Price"].dtype)
print("T3c - Year dtype:", df["Year"].dtype)

# Correct known inconsistent category spellings.
df["Registered City"] = df["Registered City"].replace("Sukkar", "Sukkur")
df["Model"] = df["Model"].replace("Corrolla Altis", "Corolla Altis")
print("T3d - City categories before correction:", city_categories_before)
print("T3d - City categories after correction:", df["Registered City"].nunique(dropna=True))

# Remove exact duplicate advertisements.
duplicate_rows = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print("T3e - Exact duplicate rows removed:", duplicate_rows)

# Convert mileage to numeric and mark invalid readings as missing.
df["KMs Driven"] = pd.to_numeric(df["KMs Driven"], errors="coerce")
invalid_mileage = (df["KMs Driven"].lt(100) |
                   df["KMs Driven"].gt(1_000_000) |
                   df["KMs Driven"].isin([12345, 123456]))
print("T3f - Invalid mileage readings:", int(invalid_mileage.sum()))
df.loc[invalid_mileage, "KMs Driven"] = np.nan

fake_prices = df["Price"].isin([12_345_678, 87_654_321])
brand_price_outliers = (df["Brand"].isin(["Suzuki", "Daihatsu"]) & df["Price"].gt(5_000_000))
rows_before_price_filter = len(df)
df = df.loc[~(fake_prices | brand_price_outliers)].copy()
print("T3f - Price outlier rows removed:", rows_before_price_filter - len(df))

# Remove derived columns if the supplied input was already cleaned.
for derived_column in ["Price_lakh", "Age"]:
    if derived_column in df.columns:
        df = df.drop(columns=derived_column)

print("T3g - Final cleaned shape:", df.shape)
print("T3g - Remaining duplicate rows:", int(df.duplicated().sum()))
assert df["Price"].notna().all(), "Price must contain valid numeric values."
assert df["Price"].dtype == "int64", "Price must have int64 dtype."
assert df["Year"].dtype == "Int64", "Year must have nullable Int64 dtype."

## Task 4 · Analyse
- (a) Add `Price_lakh`.
- (b) Add `Age` using 2020 as the reference year.
- (c) Show the 5 most common brands.
- (d) Show the 5 cities with the most advertisements.
- (e) Show median price in lakh for each fuel type, highest first.
- (f) Show Corolla GLI median price for each year from 2010 to 2018 and estimate the average yearly difference when both endpoint years are available.
- (g) Show the 5 most expensive advertisements.
- (h) Compare advertisement count and median price by transaction type.

In [ ]:
df["Price_lakh"] = df["Price"] / 100_000
print("T4a - Price in lakh:")
display(df[["Price", "Price_lakh"]].head())

df["Age"] = 2020 - df["Year"]
print("T4b - Vehicle age:")
display(df[["Year", "Age"]].head())

print("T4c - Five most common brands:")
display(df["Brand"].value_counts().head(5))

print("T4d - Five cities with the most advertisements:")
display(df["Registered City"].value_counts().head(5))

print("T4e - Median price by fuel type (lakh):")
display(df.groupby("Fuel", dropna=False)["Price_lakh"].median().sort_values(ascending=False))

print("T4f - Corolla GLI median price by year:")
corolla_gli = df.loc[df["Model"].astype("string").str.casefold().eq("corolla gli")].copy()
corolla_gli["Year"] = pd.to_numeric(corolla_gli["Year"], errors="coerce")
corolla_gli = corolla_gli.loc[corolla_gli["Year"].between(2010, 2018)]
median_by_year = corolla_gli.groupby("Year")["Price_lakh"].median().sort_index()
display(median_by_year)
if 2010 in median_by_year.index and 2018 in median_by_year.index:
    average_difference = (median_by_year.loc[2018] - median_by_year.loc[2010]) / 8
    print("Average difference per model year (lakh):", round(float(average_difference), 2))
else:
    print("Average difference not calculated because 2010 or 2018 data is unavailable.")

print("T4g - Five highest-priced advertisements:")
display(df.nlargest(5, "Price")[["Brand", "Model", "Year", "Price_lakh"]])

print("T4h - Transaction type comparison:")
transaction_summary = df.groupby("Transaction Type", dropna=False).agg(
    Advertisement_Count=("Price", "size"),
    Median_Price_Lakh=("Price_lakh", "median")
)
display(transaction_summary)

## Task 5 · Save your work
Save the cleaned dataset as `pakistan_used_cars_clean.csv` and print its shape.

In [ ]:
output_file = Path("pakistan_used_cars_clean.csv")
df.to_csv(output_file, index=False)
print("T5 - Cleaned dataset saved to:", output_file.resolve())
print("T5 - Exported dataset shape:", df.shape)

## Task 6 · Reflection
The most misleading issue would be incorrect price formatting because text-formatted prices cannot be compared or summarized correctly as numbers. I learned that advertisements are distributed differently across brands and cities. I also learned that fuel type and transaction type can be compared using median advertised prices. These results describe advertisements in the dataset, not confirmed vehicle sales.